## Databricks notebook: Auto Loader practice with JSON orders
## ---------------------------------------------------------

### Scenario:
This notebook demonstrates the **Medallion Architecture** (Bronze → Silver → Gold) using **Auto Loader** for incremental JSON file ingestion.

**Problem Statement:**
- JSON order files continuously land in a cloud path (or Unity Catalog volume)
- Schema evolves over time: new fields appear, data types change, late-arriving updates occur
- Need to handle schema evolution gracefully without breaking the pipeline

**Solution Architecture:**
- **Bronze Layer**: Raw ingestion with Auto Loader using `schemaEvolutionMode=rescue` to capture unexpected fields in `_rescued_data`
- **Silver Layer**: Cleaned, validated, and deduplicated data with proper typing and quality checks
- **Gold Layer**: Aggregated business metrics ready for analytics

**Key Auto Loader Features Used:**
- `cloudFiles.format`: JSON file format
- `cloudFiles.inferColumnTypes`: Automatic type inference
- `cloudFiles.schemaLocation`: Schema registry for evolution tracking
- `cloudFiles.schemaEvolutionMode=rescue`: Handles new/unexpected fields gracefully
- `rescuedDataColumn`: Stores fields that don't match current schema

In [0]:
###1) Paths (adjust to your environment)
### For Unity Catalog volumes, use paths like:
source_path =  "/Volumes/training/my_schema/my_volume/autoloader/json/input/"
schema_path = "/Volumes/training/my_schema/my_volume/autoloader/json/schema/"
checkpoint_path = "/Volumes/training/my_schema/my_volume/autoloader/json/checkpoint/"


In [0]:
### 2) Bronze ingestion with Auto Loader
from pyspark.sql.functions import current_timestamp, col

bronze_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("rescuedDataColumn", "_rescued_data")
    .option("cloudFiles.maxFilesPerTrigger", "100")  # small batches for practice
    .load(source_path)
    .withColumn("_ingest_time", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)


In [0]:
# Write to Bronze Delta table
(bronze_df.writeStream
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("training.auto_loader_demo.orders_bronze")
)


In [0]:
# 3) Silver: clean, validate, and deduplicate
# Run this as a separate notebook/job after Bronze is populated.

from pyspark.sql.functions import col, to_timestamp, when, row_number
from pyspark.sql.window import Window

silver_df = (
    spark.table("training.auto_loader_demo.orders_bronze")
    # Parse and validate event_time
    .withColumn("event_time_ts", to_timestamp(col("event_time")))
    # Basic quality flags
    .withColumn(
        "is_valid_order",
        (col("order_id").isNotNull()) &
        (col("customer_id").isNotNull()) &
        (col("amount").isNotNull()) &
        (col("status").isNotNull()) &
        (col("event_time_ts").isNotNull())
    )
    # Cast amount to double in case it came as string
    .withColumn("amount_num", col("amount").cast("double"))
)



In [0]:
# Deduplicate by order_id, keeping the latest event_time (or latest _ingest_time)
# Optimized: filter invalid orders first, then deduplicate only valid records
silver_dedup = (
    silver_df
    .filter(col("is_valid_order") == True)  # filter early to reduce data volume
    .withColumn(
        "rn", 
        row_number().over(
            Window.partitionBy("order_id")
            .orderBy(col("event_time_ts").desc_nulls_last(), col("_ingest_time").desc())
        )
    )
    .filter(col("rn") == 1)
    .select(
        "order_id",
        "customer_id",
        "amount_num",
        "currency",
        "status",
        "event_time_ts",
        "discount",
        "items",
        "source",
        "_source_file",
        "_ingest_time"
    )
)


In [0]:
# Write Silver as a batch overwrite (for practice; in prod you might use MERGE)
# Optimized: filter invalid orders early to reduce shuffle volume
from pyspark.sql.window import Window
from pyspark.sql.functions import col, to_timestamp, row_number

silver_dedup_fixed = (
    spark.table("training.auto_loader_demo.orders_bronze")
    .withColumn("event_time_ts", to_timestamp(col("event_time")))
    .withColumn("amount_num", col("amount").cast("double"))
    # Filter early to reduce data volume before shuffle
    .filter(
        col("order_id").isNotNull() &
        col("customer_id").isNotNull() &
        col("amount").isNotNull() &
        col("status").isNotNull() &
        col("event_time_ts").isNotNull()
    )
    # Deduplicate on reduced dataset
    .withColumn(
        "rn", 
        row_number().over(
            Window.partitionBy("order_id")
            .orderBy(col("event_time_ts").desc_nulls_last(), col("_ingest_time").desc())
        )
    )
    .filter(col("rn") == 1)
    .select(
        "order_id",
        "customer_id",
        "amount_num",
        "currency",
        "status",
        "event_time_ts",
        "_source_file",
        "_ingest_time"
    )
)

silver_dedup_fixed.write.mode("overwrite").saveAsTable("training.auto_loader_demo.orders_silver")


In [0]:
# 5) Optional: inspect rescued data
rescued_df = (
    spark.table("training.auto_loader_demo.orders_bronze")
    .filter(col("_rescued_data").isNotNull())
    .select("order_id", "customer_id", "_rescued_data", "_source_file", "_ingest_time")
)
display(rescued_df)

## Handling Rescued Data from Schema Evolution

**What is Rescued Data?**
When Auto Loader encounters fields that don't match the current schema (new fields, type mismatches, unexpected structure), it stores them in the `_rescued_data` column as JSON instead of failing the ingestion.

**Why Handle Rescued Data?**
- **Schema Evolution Detection**: Identify when source systems add new fields
- **Data Quality**: Catch type mismatches and structural changes early
- **No Data Loss**: Preserve all incoming data even if it doesn't fit the current schema
- **Backward Compatibility**: Continue processing while investigating schema changes

**The following cells demonstrate:**
1. Querying records with rescued data to identify schema evolution
2. Parsing the rescued JSON to extract specific fields
3. Incorporating rescued fields back into the main schema
4. Defining explicit schemas for commonly rescued fields
5. Writing rescued data to a separate table for further analysis

In [0]:
# Handle rescued data from Auto Loader
# Extract and parse the _rescued_data column to understand schema evolution issues

from pyspark.sql.functions import col, from_json, schema_of_json, get_json_object

# 1) Read the bronze table and filter for records with rescued data
rescued_records = (
    spark.table("training.auto_loader_demo.orders_bronze")
    .filter(col("_rescued_data").isNotNull())
)

# 2) Display rescued records to inspect what fields were rescued
print("Records with rescued data:")
display(rescued_records.select(
    "order_id", 
    "customer_id", 
    "_rescued_data", 
    "_source_file", 
    "_ingest_time"
))

# 3) Parse specific fields from the rescued data JSON
# Example: extract a field called 'new_field' that might have been added later
rescued_parsed = (
    rescued_records
    .withColumn("rescued_new_field", get_json_object(col("_rescued_data"), "$.new_field"))
    .withColumn("rescued_extra_data", get_json_object(col("_rescued_data"), "$.extra_data"))
)

display(rescued_parsed.select(
    "order_id", 
    "_rescued_data",
    "rescued_new_field",
    "rescued_extra_data"
))

# 4) Optional: Merge rescued fields back into the main schema
# After identifying rescued fields, you can update your schema or reprocess
final_df = (
    spark.table("training.auto_loader_demo.orders_bronze")
    .withColumn("new_field", 
                when(col("_rescued_data").isNotNull(), 
                     get_json_object(col("_rescued_data"), "$.new_field"))
                .otherwise(None))
)

print("\nFinal DataFrame with rescued fields incorporated:")
display(final_df)

## Advanced Rescued Data Processing

**Purpose**: Extract and transform rescued data into structured tables

**Process Flow:**
1. **Define Schema** (Cell 12): Create explicit schema for known rescued fields (discount, source, items, _file_path)
2. **Parse JSON** (Cell 13): Use `from_json()` to convert rescued JSON into structured columns
3. **Explode Arrays** (Cell 14): Flatten array fields (like items) into individual rows
4. **Clean & Transform** (Cell 15): Remove unnecessary columns and prepare for storage
5. **Persist** (Cell 17): Save to `orders_bronze_mismatch` table for analysis

**When to Use This Pattern:**
- Rescued data appears consistently (indicates schema drift in source)
- Need to incorporate rescued fields into downstream processing
- Want to analyze schema evolution patterns over time
- Preparing to update the main schema to include new fields

In [0]:
from pyspark.sql.types import StructType, StructField, DoubleType, StringType, ArrayType

rescue_schema = StructType([
    StructField("discount", DoubleType(), True),
    StructField("source", StringType(), True),
    StructField("items", ArrayType(StringType()), True),
    StructField("_file_path", StringType(), True)
])

In [0]:
# Redefine rescue_df with corrected logic
from pyspark.sql import functions as F
rescue_df_fixed = (
    final_df
    .withColumn("rescued", F.from_json(F.col("_rescued_data"), rescue_schema))
    .select(
        F.col("order_id").alias("order_id"),
        F.col("rescued.discount").alias("discount"),
        F.col("rescued.source").alias("source"),
        F.col("rescued.items").alias("items"),
        F.col("rescued._file_path").alias("file_path")
    ).filter(col("_rescued_data").isNotNull())

)
#display(rescue_df_fixed)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
rescue_df_fixed_df = rescue_df_fixed.withColumn("Items",explode(col("items")))
#display(rescue_df_fixed_df)

In [0]:
 
rescue_df_fixed_1 = (
    rescue_df_fixed_df
    .drop(
        "amount",
        "currency",
        "customer_id",
        "event_time",
        "status",
        "_rescued_data",
        "_ingest_time",
        "_source_file",
        "new_field",
        "_file_path"
    )
)

In [0]:
%sql
--drop table training.auto_loader_demo.orders_bronze_mismatch;

In [0]:
rescue_df_fixed_1.write.mode("append").format("delta").saveAsTable("training.auto_loader_demo.orders_bronze_mismatch")

In [0]:
%sql
create table if not exists training.auto_loader_demo.orders_gold
as 
select  customer_id, sum(amount_num)  as total_amount, status
from training.auto_loader_demo.orders_silver
group by customer_id,status
order by customer_id,total_amount;